In [3]:
from forex_python.converter import CurrencyRates

def calculate_lot_size_auto(
    balance,
    risk_percent,
    entry_price,
    stop_loss_price,
    pair,
    account_currency="USD"
):
    """
    Universal lot size calculator for any forex pair.
    Automatically fetches live exchange rates via forex-python.

    balance:          account balance in account currency
    risk_percent:     risk per trade (e.g. 2 for 2%)
    entry_price:      trade entry price
    stop_loss_price:  stop loss price
    pair:             e.g. "USD/CAD", "EUR/JPY", "GBP/ZAR"
    account_currency: your account currency (default = USD)
    """

    c = CurrencyRates()
    base, quote = pair.split("/")

    # 1️⃣ Pip size (0.01 for JPY pairs, otherwise 0.0001)
    pip_size = 0.01 if "JPY" in quote else 0.0001

    # 2️⃣ Pip value in the quote currency (before conversion)
    if quote == account_currency:
        pip_value = pip_size * 100000
    elif base == account_currency:
        pip_value = (pip_size / entry_price) * 100000
    else:
        # Pip value in quote currency
        pip_value_quote = (pip_size / entry_price) * 100000
        # Convert to account currency
        conversion_rate = c.get_rate(quote, account_currency)
        pip_value = pip_value_quote * conversion_rate

    # 3️⃣ Risk amount
    risk_amount = balance * (risk_percent / 100)

    # 4️⃣ Stop loss distance in pips
    pip_distance = abs(entry_price - stop_loss_price) / pip_size

    # 5️⃣ Lot size
    lot_size = risk_amount / (pip_distance * pip_value)
    
    def calculate_required_margin(lot_size, pair, entry_price, leverage):
        """
        Calculates the margin required to open a position.
        Assumes 1 lot = 100,000 units of base currency.
        """
        base, quote = pair.split("/")
        position_value = lot_size * 100000 * entry_price  # value in quote currency
        margin_required = position_value / leverage
        return margin_required

    margin_required = calculate_required_margin(lot_size, pair, entry_price, 200)
    
    return lot_size, margin_required


In [4]:
balance = 100
risk_percent = 1
entry_price = 1.15623
stop_loss_price = 1.15612
# pair = "USD/CAD"
pair = "EUR/USD"

lot_size, margin_required = calculate_lot_size_auto(balance, risk_percent, entry_price, stop_loss_price, pair)
print(f"Lot size: {lot_size:.3f} and margin required: {margin_required:.2f} USD")

Lot size: 0.091 and margin required: 52.56 USD


In [ ]:
  starting_balance: 100
  risk_per_trade: 0.01
  spread_pips: 0.2
  commission_per_trade: 0.0
  leverage: 100
  max_drawdown_stop_pct: 0   # stop trading after 30% drawdown
  slippage_pips: 0.1          # 0.1 pips unfavorable slippage per fill
  min_stop_pips: 0            # ignore SLs tighter than 3 pips
  min_size: 0.01               # skip trades smaller than this size (enforce minimum lot 0.01)
  lot_step: 0.01               # round position size down to nearest 0.01 lot increment
  max_lot_size: 0.01           # hard cap on position size (after leverage & rounding); set 0 or omit to disable
  contract_size: 100000        # units per 1.0 lot (forex standard)
  pip_value_per_lot: 10.0      # USD value per pip for 1 lot on EURUSD-like pairs (override as needed)
  equity_rounding: 0.01        # round equity to the nearest cent for realism


In [17]:
price = 1.15629
lot_size = 0.01
current_balance = 100
leverage = 100
11.56

11.56

In [18]:
margin_used = lot_size * 100000 * price / leverage
margin_used

11.562899999999999

In [19]:
lot_size = margin_used * leverage / (100000 * price)
lot_size

0.01

In [20]:
max_lots = current_balance * leverage / (100000 * price)
max_lots

0.08648349462505081

### Download history

In [1]:
import MetaTrader5 as mt5
from datetime import datetime, timezone, timedelta
import pandas as pd
import os

LOGIN     = int(os.getenv("MT5_LOGIN", "123456"))   # fill in
PASSWORD  = os.getenv("MT5_PASSWORD")               # fill in
SERVER    = os.getenv("MT5_SERVER")                 # fill in

if not mt5.initialize():
    raise RuntimeError(f"initialize failed: {mt5.last_error()}")

if not mt5.login(LOGIN, PASSWORD, SERVER):
    raise RuntimeError(f"login failed: {mt5.last_error()}")

# Choose range (e.g., last 30 days)
to_dt   = datetime.now(timezone.utc)
from_dt = to_dt - timedelta(days=30)

# Fetch deals
deals = mt5.history_deals_get(from_dt, to_dt)
orders = mt5.history_orders_get(from_dt, to_dt)

def _records_to_df(recs):
    if recs is None:
        return pd.DataFrame()
    rows = [r._asdict() for r in recs]
    return pd.DataFrame(rows)

df_deals = _records_to_df(deals)
df_orders = _records_to_df(orders)

# Optional: sort and normalize times
for df in (df_deals, df_orders):
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)
    if 'time_setup' in df.columns:
        df['time_setup'] = pd.to_datetime(df['time_setup'], unit='s', utc=True)

df_deals.sort_values('time', inplace=True)

# Save
df_deals.to_csv("account_deals_history.csv", index=False)
df_orders.to_csv("account_orders_history.csv", index=False)

print(f"Deals rows: {len(df_deals)}, Orders rows: {len(df_orders)}")

mt5.shutdown()

Deals rows: 59, Orders rows: 34


True

In [43]:
df_orders

,ticket,time_setup,time_setup_msc,time_done,time_done_msc,time_expiration,type,type_time,type_filling,state,...,volume_initial,volume_current,price_open,sl,tp,price_current,price_stoplimit,symbol,comment,external_id
0,1946904930,2025-11-17 06:09:54+00:00,1763359794666,1763359794,1763359794753,0,0,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15962,0.0,EURUSDm,Test trade from,
1,1946909363,2025-11-17 06:10:59+00:00,1763359859779,1763359859,1763359859859,0,1,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15962,0.0,EURUSDm,,
2,1947497356,2025-11-17 08:13:57+00:00,1763367237972,1763367238,1763367238044,0,0,0,0,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.16111,0.0,EURUSDm,,
3,1947497445,2025-11-17 08:13:59+00:00,1763367239647,1763367239,1763367239722,0,1,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.16108,0.0,EURUSDm,,
4,1948171719,2025-11-17 10:03:00+00:00,1763373780996,1763373781,1763373781069,0,1,0,1,4,...,0.02,0.0,0.00000,1.16014,1.15615,1.16003,0.0,EURUSDm,bot-test,
5,1948172281,2025-11-17 10:03:06+00:00,1763373786193,1763373786,1763373786274,0,0,0,1,4,...,0.02,0.0,0.00000,0.00000,0.00000,1.16013,0.0,EURUSDm,,
6,1956153167,2025-11-18 12:34:48+00:00,1763469288851,1763469288,1763469288939,0,0,0,0,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15899,0.0,EURUSDm,,
7,1956159185,2025-11-18 12:35:38+00:00,1763469338680,1763469338,1763469338765,0,1,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15891,0.0,EURUSDm,,
8,1963036280,2025-11-19 13:26:31+00:00,1763558791290,1763558791,1763558791373,0,0,0,1,4,...,0.02,0.0,0.00000,1.15762,1.15974,1.15866,0.0,EURUSDm,test_trade,
9,1963057374,2025-11-19 13:30:02+00:00,1763559002688,1763559002,1763559002783,0,1,0,1,4,...,0.02,0.0,0.00000,0.00000,0.00000,1.15826,0.0,EURUSDm,,


In [41]:
df_pnl = df_orders[df_orders['time_setup'] > "2025-11-18 12:30:00+00:00"][['time_setup', 'profit']]

KeyError: "['profit'] not in index"

In [33]:
df_orders

,ticket,time_setup,time_setup_msc,time_done,time_done_msc,time_expiration,type,type_time,type_filling,state,...,volume_initial,volume_current,price_open,sl,tp,price_current,price_stoplimit,symbol,comment,external_id
0,1946904930,2025-11-17 06:09:54+00:00,1763359794666,1763359794,1763359794753,0,0,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15962,0.0,EURUSDm,Test trade from,
1,1946909363,2025-11-17 06:10:59+00:00,1763359859779,1763359859,1763359859859,0,1,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15962,0.0,EURUSDm,,
2,1947497356,2025-11-17 08:13:57+00:00,1763367237972,1763367238,1763367238044,0,0,0,0,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.16111,0.0,EURUSDm,,
3,1947497445,2025-11-17 08:13:59+00:00,1763367239647,1763367239,1763367239722,0,1,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.16108,0.0,EURUSDm,,
4,1948171719,2025-11-17 10:03:00+00:00,1763373780996,1763373781,1763373781069,0,1,0,1,4,...,0.02,0.0,0.00000,1.16014,1.15615,1.16003,0.0,EURUSDm,bot-test,
5,1948172281,2025-11-17 10:03:06+00:00,1763373786193,1763373786,1763373786274,0,0,0,1,4,...,0.02,0.0,0.00000,0.00000,0.00000,1.16013,0.0,EURUSDm,,
6,1956153167,2025-11-18 12:34:48+00:00,1763469288851,1763469288,1763469288939,0,0,0,0,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15899,0.0,EURUSDm,,
7,1956159185,2025-11-18 12:35:38+00:00,1763469338680,1763469338,1763469338765,0,1,0,1,4,...,0.01,0.0,0.00000,0.00000,0.00000,1.15891,0.0,EURUSDm,,
8,1963036280,2025-11-19 13:26:31+00:00,1763558791290,1763558791,1763558791373,0,0,0,1,4,...,0.02,0.0,0.00000,1.15762,1.15974,1.15866,0.0,EURUSDm,test_trade,
9,1963057374,2025-11-19 13:30:02+00:00,1763559002688,1763559002,1763559002783,0,1,0,1,4,...,0.02,0.0,0.00000,0.00000,0.00000,1.15826,0.0,EURUSDm,,


### Download historical data

In [11]:
# Quick historical FX data downloader via MetaTrader5
from datetime import datetime, timezone, timedelta
import pandas as pd
import os

try:
    import MetaTrader5 as mt5
except ImportError as e:
    raise RuntimeError("MetaTrader5 package not installed. Install with: pip install MetaTrader5 (Windows only).")

# ---- CONFIG ----
SYMBOL = "EURUSDm"        # Change as needed (e.g. "GBPUSD", "USDJPY")
TIMEFRAME = "5m"          # Supported: 1m,5m,15m,30m,1h,4h,1d
DAYS_BACK = 4             # How many days of history to pull
OUTPUT_CSV = f"history_{SYMBOL}_{TIMEFRAME}_{DAYS_BACK}d.csv"

LOGIN     = int(os.getenv("MT5_LOGIN", "0")) or None
PASSWORD  = os.getenv("MT5_PASSWORD") or None
SERVER    = os.getenv("MT5_SERVER") or None

# Timeframe mapping for MetaTrader5 constants
TF_MAP = {
    '1m': mt5.TIMEFRAME_M1,
    '5m': mt5.TIMEFRAME_M5,
    '15m': mt5.TIMEFRAME_M15,
    '30m': mt5.TIMEFRAME_M30,
    '1h': mt5.TIMEFRAME_H1,
    '4h': mt5.TIMEFRAME_H4,
    '1d': mt5.TIMEFRAME_D1,
}
if TIMEFRAME not in TF_MAP:
    raise ValueError(f"Unsupported timeframe {TIMEFRAME}")

# ---- INITIALIZE ----
if not mt5.initialize():
    raise RuntimeError(f"initialize failed: {mt5.last_error()}")
if LOGIN and PASSWORD and SERVER:
    if not mt5.login(LOGIN, PASSWORD, SERVER):
        raise RuntimeError(f"login failed: {mt5.last_error()}")

# Ensure symbol visible
info = mt5.symbol_info(SYMBOL)
if info is None:
    raise RuntimeError(f"Symbol {SYMBOL} not found in terminal.")
if not info.visible:
    mt5.symbol_select(SYMBOL, True)

# ---- FETCH RANGE ----
end = datetime.now(timezone.utc)
start = end - timedelta(days=DAYS_BACK)

rates = mt5.copy_rates_range(SYMBOL, TF_MAP[TIMEFRAME], start, end)
if rates is None:
    raise RuntimeError(f"Failed to fetch rates: {mt5.last_error()}")

# Convert to DataFrame
bars = pd.DataFrame(rates)
bars['time'] = pd.to_datetime(bars['time'], unit='s', utc=True)
bars = bars.rename(columns={
    'time': 'Date', 'open': 'Open', 'high': 'High', 'low': 'Low',
    'close': 'Close', 'tick_volume': 'Volume'
})
bars = bars[['Date','Open','High','Low','Close','Volume']].set_index('Date').sort_index()

# ---- SAVE ----
bars.to_csv(OUTPUT_CSV)
print(f"Saved {len(bars)} bars to {OUTPUT_CSV}")
print(bars.head())

mt5.shutdown()

Saved 1148 bars to history_EURUSDm_5m_4d.csv
                              Open     High      Low    Close  Volume
Date                                                                 
2025-11-17 05:55:00+00:00  1.15998  1.16003  1.15988  1.16001      88
2025-11-17 06:00:00+00:00  1.16003  1.16003  1.15989  1.15992      76
2025-11-17 06:05:00+00:00  1.15991  1.15991  1.15948  1.15953     157
2025-11-17 06:10:00+00:00  1.15951  1.15976  1.15948  1.15971      88
2025-11-17 06:15:00+00:00  1.15969  1.15984  1.15962  1.15983      84


True

In [29]:
bars_list = []
for row in bars.iterrows():
    bars_list.append({"Date": str(row[0]), 
                        "Open": float(row[1]['Open']),
                        "High": float(row[1]['High']),
                        "Low": float(row[1]['Low']),
                        "Close": float(row[1]['Close']),
                        "Volume": float(row[1]['Volume'])})

In [ ]:
import json

bars_list

[{'Date': '2025-11-17 05:55:00+00:00',
  'Open': 1.15998,
  'High': 1.16003,
  'Low': 1.15988,
  'Close': 1.16001,
  'Volume': 88.0},
 {'Date': '2025-11-17 06:00:00+00:00',
  'Open': 1.16003,
  'High': 1.16003,
  'Low': 1.15989,
  'Close': 1.15992,
  'Volume': 76.0},
 {'Date': '2025-11-17 06:05:00+00:00',
  'Open': 1.15991,
  'High': 1.15991,
  'Low': 1.15948,
  'Close': 1.15953,
  'Volume': 157.0},
 {'Date': '2025-11-17 06:10:00+00:00',
  'Open': 1.15951,
  'High': 1.15976,
  'Low': 1.15948,
  'Close': 1.15971,
  'Volume': 88.0},
 {'Date': '2025-11-17 06:15:00+00:00',
  'Open': 1.1596899999999999,
  'High': 1.15984,
  'Low': 1.15962,
  'Close': 1.15983,
  'Volume': 84.0},
 {'Date': '2025-11-17 06:20:00+00:00',
  'Open': 1.15984,
  'High': 1.1598600000000001,
  'Low': 1.15975,
  'Close': 1.15984,
  'Volume': 83.0},
 {'Date': '2025-11-17 06:25:00+00:00',
  'Open': 1.15985,
  'High': 1.16005,
  'Low': 1.15984,
  'Close': 1.16003,
  'Volume': 89.0},
 {'Date': '2025-11-17 06:30:00+00:00',


In [31]:
# Save bars_list (list of dicts) to JSON and read it back
import json
from pathlib import Path

OUTPUT_JSON = Path("bars_history.json")

# bars_list is expected to be defined earlier (list of dictionaries)
if 'bars_list' not in globals():
    raise NameError("bars_list not found. Run the data collection cell first.")

# Write JSON with pretty formatting
with OUTPUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(bars_list, f, indent=2, ensure_ascii=False)

print(f"Saved {len(bars_list)} records to {OUTPUT_JSON}")

# Read back to verify
with OUTPUT_JSON.open('r', encoding='utf-8') as f:
    reloaded = json.load(f)

print(f"Reloaded {len(reloaded)} records. First record snippet:")
print(reloaded[0] if reloaded else 'No data')

Saved 1148 records to bars_history.json
Reloaded 1148 records. First record snippet:
{'Date': '2025-11-17 05:55:00+00:00', 'Open': 1.15998, 'High': 1.16003, 'Low': 1.15988, 'Close': 1.16001, 'Volume': 88.0}
